# 403 — Epigenetic Regulator Enrichment

## Objective

Characterize whether curated epigenetic regulators and epigenetic regulatory
complexes are preferentially concentrated among high-weight genes of the three
frozen cross-system consensus transcriptomic representations constructed in
notebook 401.

The analysis is restricted to the frozen 2,389-gene consensus universe and uses
EpiFactors v2.1 as the external curated annotation source.

## Analytical status

Notebook 403 is a biological-interpretation layer applied to the frozen Phase 4
consensus representations.

The identities, gene weights, orientations, source-program mappings, tumor-side
methylation context, and cross-lineage robustness results of the three consensus
programs are upstream constraints and are not redefined here.

Notebook 402 does not act as an eligibility gate for 403. All three frozen
consensus representations enter the enrichment analysis.

## Enrichment framework

The primary analysis evaluates prespecified epigenetic-regulator classes using
rank-based enrichment over the absolute consensus weights.

The statistical background is the frozen 2,389-gene consensus universe rather
than the full genome or the complete EpiFactors catalog.

Protein-complex enrichment is treated as an exploratory secondary analysis.
DNMT, TET, HDAC, and KDM families are inspected descriptively rather than forced
into small-set enrichment tests.

Source-system tumor and cell-line loadings are retained as contextual evidence
to assess whether regulator-associated signal is represented symmetrically or
is concentrated in one source system.

## Scope and methodological boundary

This notebook does not:

- redefine consensus-program membership or gene weights;
- use epigenetic enrichment to rescue or exclude a consensus program;
- introduce pharmacogenomic phenotype information;
- infer regulator activity from expression;
- construct causal regulatory networks;
- interpret enrichment as evidence of a master regulator or causal mechanism;
- use tumor methylation information to alter enrichment results; or
- introduce additional annotation resources after inspecting the results.

Enrichment and regulator-level inspection are interpretive analyses only.
They may support biological contextualization and candidate associated-regulator
hypotheses, but they do not establish causality, mechanistic validation,
clinical relevance, or therapeutic actionability.

## Expected outputs

Notebook 403 will publish two downstream-consumable artifacts under
`data/processed/consensus_programs/`:

- `403_epigenetic_regulator_enrichment_summary.csv` — program-level enrichment
  results for prespecified regulator classes and eligible exploratory epigenetic
  complexes.
- `403_epigenetic_regulator_gene_context.csv` — gene-level EpiFactors annotation
  and consensus/source-system loading context for epigenetic regulators present
  in the frozen consensus universe.

Permutation null distributions, intermediate gene-set tables, descriptive
family subsets, metadata files, and figures are not persisted as separate
artifacts.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import numpy as np
import pandas as pd

from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.utils.paths import Paths

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

EPIFACTORS_DIR = Paths.epifactors
CONSENSUS_PROGRAM_DIR = Paths.consensus_programs
OUTPUT_DIR = Paths.consensus_programs

In [3]:
# =============================================================================
# Authoritative epigenetic-regulator enrichment input paths
# =============================================================================

CONSENSUS_CATALOG_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_program_catalog.csv"
)

CONSENSUS_GENE_WEIGHTS_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_gene_weights.csv"
)

EPIFACTORS_PROTEINS_PATH = (
    EPIFACTORS_DIR
    / "EpiGenes_main.csv"
)

EPIFACTORS_COMPLEXES_PATH = (
    EPIFACTORS_DIR
    / "EpiGenes_complexes.csv"
)

In [4]:
# =============================================================================
# Load authoritative epigenetic-regulator enrichment inputs
# =============================================================================

consensus_catalog = pd.read_csv(CONSENSUS_CATALOG_PATH)

consensus_gene_weights = pd.read_csv(
    CONSENSUS_GENE_WEIGHTS_PATH
)

epifactors_proteins = pd.read_csv(
    EPIFACTORS_PROTEINS_PATH
)

epifactors_complexes = pd.read_csv(
    EPIFACTORS_COMPLEXES_PATH
)

In [5]:
# =============================================================================
# Inspect EpiFactors input schemas
# =============================================================================

{
    "proteins_shape": epifactors_proteins.shape,
    "proteins_columns": epifactors_proteins.columns.tolist(),
    "complexes_shape": epifactors_complexes.shape,
    "complexes_columns": epifactors_complexes.columns.tolist(),
}

{'proteins_shape': (801, 25),
 'proteins_columns': ['Id',
  'HGNC_symbol',
  'Status',
  'HGNC_ID',
  'HGNC_name',
  'GeneID',
  'UniProt_AC',
  'UniProt_ID',
  'Domain',
  'MGI_symbol',
  'MGI_ID',
  'UniProt_AC_Mm',
  'UniProt_ID_Mm',
  'GeneTag',
  'GeneDesc',
  'Function',
  'Modification',
  'PMID_function',
  'Complex_name',
  'Target',
  'Specific_target',
  'Product',
  'UniProt_ID_target',
  'PMID_target',
  'Comment'],
 'complexes_shape': (73, 17),
 'complexes_columns': ['Id',
  'Group',
  'Group_name',
  'Complex_name',
  'Status',
  'Alternative_name',
  'Protein',
  'UniProt_ID',
  'PMID_complex',
  'Function',
  'PMID_function',
  'Target',
  'Specific_target',
  'Product',
  'Uniprot_ID_target',
  'PMID_target',
  'Comment']}

In [6]:
# =============================================================================
# Inspect EpiFactors annotation encoding
# =============================================================================

{
    "protein_status": epifactors_proteins["Status"].value_counts(
        dropna=False
    ).to_dict(),
    "gene_tags": epifactors_proteins["GeneTag"].value_counts(
        dropna=False
    ).head(20).to_dict(),
    "functions": epifactors_proteins["Function"].value_counts(
        dropna=False
    ).head(20).to_dict(),
    "modifications": epifactors_proteins["Modification"].value_counts(
        dropna=False
    ).head(20).to_dict(),
    "complex_groups": epifactors_complexes["Group_name"].value_counts(
        dropna=False
    ).to_dict(),
}

{'protein_status': {'#': 720, 'New': 81},
 'gene_tags': {'#': 363,
  'RBM': 38,
  'PHF': 34,
  'WDR': 20,
  'ZNF': 20,
  'RNF': 15,
  'KMT': 13,
  'USP': 12,
  'INO80': 11,
  'APOBEC': 10,
  'METTL': 9,
  'SAMD': 9,
  'PRMT': 8,
  'KDM': 8,
  'UBE2': 8,
  'KDM, PHF': 7,
  'ANKRD': 6,
  'KMT, PHF': 6,
  'RNF, PCGF': 6,
  'CDK': 6},
 'functions': {'Histone modification write': 147,
  'Chromatin remodeling': 80,
  'Histone modification read': 80,
  'RNA modification': 75,
  'Histone modification write cofactor': 65,
  'Histone modification erase': 58,
  'Histone modification erase cofactor': 43,
  'Chromatin remodeling cofactor': 34,
  'Histone chaperone': 23,
  'Polycomb group (PcG) protein': 21,
  'Histone modification': 15,
  'DNA modification': 13,
  'TF': 12,
  'Histone modification cofactor': 11,
  'Histone modification write cofactor, Histone modification write cofactor': 10,
  'DNA modification, RNA modification': 9,
  'Chromatin remodeling, TF': 9,
  'Scaffold protein, RNA modifi

In [7]:
# =============================================================================
# Extract EpiFactors annotation vocabularies
# =============================================================================

function_tokens = (
    epifactors_proteins["Function"]
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
)

modification_tokens = (
    epifactors_proteins["Modification"]
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
)

{
    "function_tokens": sorted(
        function_tokens.loc[function_tokens.ne("#")].unique()
    ),
    "modification_tokens": sorted(
        modification_tokens.loc[modification_tokens.ne("#")].unique()
    ),
}

{'function_tokens': ['Chromatin remodeling',
  'Chromatin remodeling cofactor',
  'DNA modification',
  'DNA modification cofactor',
  'Histone chaperone',
  'Histone modification',
  'Histone modification cofactor',
  'Histone modification erase',
  'Histone modification erase cofactor',
  'Histone modification read',
  'Histone modification read cofactor',
  'Histone modification write',
  'Histone modification write cofactor',
  'Histone modification writer',
  'Polycomb group (PcG) protein',
  'Protein modification',
  'RNA modification',
  'Scaffold protein',
  'TF'],
 'modification_tokens': ['Alternative splicing',
  'DNA demethylation',
  'DNA hydroxymethylation',
  'DNA methylation',
  'Histone GlcNAcylation',
  'Histone acetylation',
  'Histone citrullination',
  'Histone deacetylation',
  'Histone deubiquitination',
  'Histone methylation',
  'Histone phosphorylation',
  'Histone sumoylation',
  'Histone ubiquitination',
  'Protein methylation',
  'RNA acetylation',
  'RNA de

In [8]:
# =============================================================================
# Freeze epigenetic-regulator enrichment parameters
# =============================================================================

MIN_SET_SIZE = 10
N_PERMUTATIONS = 10_000
RANDOM_SEED = 403

PRIMARY_REGULATOR_CLASSES = {
    "EPIFACTORS_ALL": {
        "source": "all",
        "tokens": None,
    },
    "DNA_MODIFICATION": {
        "source": "function",
        "tokens": {
            "DNA modification",
            "DNA modification cofactor",
        },
    },
    "HISTONE_ACETYLATION": {
        "source": "modification",
        "tokens": {
            "Histone acetylation",
            "Histone deacetylation",
        },
    },
    "HISTONE_METHYLATION": {
        "source": "modification",
        "tokens": {
            "Histone methylation",
        },
    },
    "CHROMATIN_REMODELING_SYSTEM": {
        "source": "function",
        "tokens": {
            "Chromatin remodeling",
            "Chromatin remodeling cofactor",
        },
    },
    "POLYCOMB_GROUP": {
        "source": "function",
        "tokens": {
            "Polycomb group (PcG) protein",
        },
    },
}

In [9]:
# =============================================================================
# Define the frozen consensus gene universe
# =============================================================================

consensus_gene_universe = (
    consensus_gene_weights[["gene_symbol"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

consensus_gene_universe.shape

(2389, 1)

In [10]:
# =============================================================================
# Harmonize EpiFactors genes with the frozen consensus universe
# =============================================================================

consensus_gene_symbols = set(
    consensus_gene_universe["gene_symbol"]
)

epifactors_gene_annotations = epifactors_proteins.copy()

epifactors_gene_annotations["gene_symbol"] = (
    epifactors_gene_annotations["HGNC_symbol"]
    .astype("string")
    .str.strip()
)

epifactors_gene_annotations = epifactors_gene_annotations.loc[
    epifactors_gene_annotations["gene_symbol"].notna()
    & epifactors_gene_annotations["gene_symbol"].ne("#")
].copy()

epifactors_gene_annotations["in_consensus_universe"] = (
    epifactors_gene_annotations["gene_symbol"].isin(
        consensus_gene_symbols
    )
)

{
    "epifactors_genes": epifactors_gene_annotations["gene_symbol"].nunique(),
    "genes_in_consensus_universe": (
        epifactors_gene_annotations.loc[
            epifactors_gene_annotations["in_consensus_universe"],
            "gene_symbol",
        ].nunique()
    ),
}

{'epifactors_genes': 796, 'genes_in_consensus_universe': 32}

In [11]:
# =============================================================================
# Inspect EpiFactors–consensus symbol overlap
# =============================================================================

matched_epifactors_symbols = sorted(
    set(epifactors_gene_annotations.loc[
        epifactors_gene_annotations["in_consensus_universe"],
        "gene_symbol",
    ])
)

unmatched_epifactors_symbols = sorted(
    set(epifactors_gene_annotations.loc[
        ~epifactors_gene_annotations["in_consensus_universe"],
        "gene_symbol",
    ])
)

{
    "matched_symbols": matched_epifactors_symbols,
    "first_unmatched_symbols": unmatched_epifactors_symbols[:40],
    "first_consensus_symbols": sorted(consensus_gene_symbols)[:40],
}

{'matched_symbols': ['ACTL6B',
  'APOBEC3B',
  'CBX2',
  'CELF3',
  'CELF4',
  'EYA1',
  'EYA2',
  'EYA4',
  'FOXA1',
  'GADD45G',
  'GFI1',
  'HDAC9',
  'HLTF',
  'HMGN5',
  'HR',
  'IKZF1',
  'IKZF3',
  'KDM5D',
  'NPM2',
  'PADI1',
  'PADI2',
  'PADI3',
  'PPARGC1A',
  'PRDM8',
  'PRKCB',
  'RBM24',
  'SNAI2',
  'SP140',
  'TLE2',
  'USP44',
  'UTY',
  'ZNF711'],
 'first_unmatched_symbols': ['A1CF',
  'ACINU',
  'ACTB',
  'ACTL6A',
  'ACTR3B',
  'ACTR5',
  'ACTR6',
  'ACTR8',
  'ADNP',
  'AEBP2',
  'AICDA',
  'AIRE',
  'ALKBH1',
  'ALKBH4',
  'ALKBH5',
  'ANKRD32',
  'ANP32A',
  'ANP32B',
  'ANP32E',
  'APBB1',
  'APEX1',
  'APOBEC1',
  'APOBEC2',
  'APOBEC3A',
  'APOBEC3C',
  'APOBEC3D',
  'APOBEC3F',
  'APOBEC3G',
  'APOBEC3H',
  'ARID1A',
  'ARID1B',
  'ARID2',
  'ARID4A',
  'ARID4B',
  'ARNTL',
  'ARRB1',
  'ASF1A',
  'ASF1B',
  'ASH1L',
  'ASH2L'],
 'first_consensus_symbols': ['AADAC',
  'AARD',
  'ABCA3',
  'ABCB1',
  'ABCC2',
  'ABCC3',
  'ABCC6',
  'ABCG2',
  'ABI3BP',
  'AB

In [12]:
# =============================================================================
# Prepare token-level EpiFactors annotations
# =============================================================================

epifactors_function_annotations = (
    epifactors_gene_annotations[["gene_symbol", "Function"]]
    .assign(annotation=lambda df: df["Function"].fillna("").str.split(", "))
    .explode("annotation")
)

epifactors_function_annotations["annotation"] = (
    epifactors_function_annotations["annotation"].str.strip()
)

epifactors_modification_annotations = (
    epifactors_gene_annotations[["gene_symbol", "Modification"]]
    .assign(annotation=lambda df: df["Modification"].fillna("").str.split(", "))
    .explode("annotation")
)

epifactors_modification_annotations["annotation"] = (
    epifactors_modification_annotations["annotation"].str.strip()
)

In [13]:
# =============================================================================
# Construct and quantify primary regulator classes
# =============================================================================

primary_regulator_gene_sets = {}
primary_regulator_class_coverage = []

for class_name, specification in PRIMARY_REGULATOR_CLASSES.items():
    if specification["source"] == "all":
        class_genes = set(
            epifactors_gene_annotations["gene_symbol"]
        )

    elif specification["source"] == "function":
        class_genes = set(
            epifactors_function_annotations.loc[
                epifactors_function_annotations["annotation"].isin(
                    specification["tokens"]
                ),
                "gene_symbol",
            ]
        )

    else:
        class_genes = set(
            epifactors_modification_annotations.loc[
                epifactors_modification_annotations["annotation"].isin(
                    specification["tokens"]
                ),
                "gene_symbol",
            ]
        )

    genes_in_universe = class_genes & consensus_gene_symbols
    primary_regulator_gene_sets[class_name] = genes_in_universe

    primary_regulator_class_coverage.append({
        "annotation_name": class_name,
        "source_gene_count": len(class_genes),
        "genes_in_consensus_universe": len(genes_in_universe),
        "coverage_fraction": len(genes_in_universe) / len(class_genes),
        "eligible_for_enrichment": len(genes_in_universe) >= MIN_SET_SIZE,
    })

primary_regulator_class_coverage = pd.DataFrame(
    primary_regulator_class_coverage
)

primary_regulator_class_coverage

,annotation_name,source_gene_count,genes_in_consensus_universe,coverage_fraction,eligible_for_enrichment
0,EPIFACTORS_ALL,796,32,0.040201,True
1,DNA_MODIFICATION,29,1,0.034483,False
2,HISTONE_ACETYLATION,142,2,0.014085,False
3,HISTONE_METHYLATION,129,4,0.031008,False
4,CHROMATIN_REMODELING_SYSTEM,145,7,0.048276,False
5,POLYCOMB_GROUP,29,0,0.000000,False


In [14]:
# =============================================================================
# Inspect EpiFactors complex membership encoding
# =============================================================================

epifactors_complexes[
    [
        "Group_name",
        "Complex_name",
        "Protein",
        "UniProt_ID",
        "Function",
    ]
].head(15)

,Group_name,Complex_name,Protein,UniProt_ID,Function
0,ISWI,ACF,"BAZ1A, SMARCA5/SNF2H(ISWI-type ATPase)","BAZ1A_HUMAN, SMCA5_HUMAN",chromatin remodeling complex
1,ISWI,B-WICH,"BAZ1B/WSTF, DDX21, DEK, ERCC6, MYBBP1A, MYO1C,...","BAZ1B_HUMAN, DDX21_HUMAN, DEK_HUMAN, ERCC6_HUM...","histone phosphorylation complex, chromatin rem..."
2,ISWI,RSF,"SMARCA5/SNF2H(ISWI-type ATPase), RSF1","RSF1_HUMAN, SMCA5_HUMAN",chromatin remodeling complex
3,ISWI,CHRAC,"CHRAC1, POLE3, ACF1 and ISWI/SNF2H","CHRC1_HUMAN, DPOE3_HUMAN, BAZ1A_HUMAN, SMCA5_H...",chromatin remodeling complex
4,ISWI,NoRC,"SMARCA5/SNF2H(ISWI-type ATPase), BAZ2A/TIP5","BAZ2A_HUMAN, SMCA5_HUMAN",chromatin remodeling complex
5,ISWI,NuRF,SMARCA1; BPTF; RBBP4 and RBBP7,"BPTF_HUMAN, RBBP4_HUMAN, RBBP7_HUMAN, SMCA1_HUMAN",chromatin remodeling complex
6,ISWI,CERF,"CECR2, SMCA1","CECR2_HUMAN, SMCA1_HUMAN",chromatin remodeling complex
7,SWI/SNF,BAF,"(ACTB), ARID1A, ARID1B/BAF250, SMARCA2, SMARCA...","ACL6A_HUMAN, ACL6B_HUMAN, ACTB_HUMAN, ARI1A_HU...",chromatin remodeling complex
8,SWI/SNF,nBAF,"ARID1A/BAF250A or ARID1B/BAF250B, SMARCD1/BAF6...","ACL6B_HUMAN, ACTB_HUMAN, (ARI1A_HUMAN|ARI1B_HU...",chromatin remodeling complex
9,SWI/SNF,npBAF,"ARID1A/BAF250A or ARID1B/BAF250B, SMARCD1/BAF6...","ACL6A_HUMAN, ACTB_HUMAN, (ARI1A_HUMAN|ARI1B_HU...",chromatin remodeling complex


In [15]:
# =============================================================================
# Prepare EpiFactors UniProt-to-HGNC mapping
# =============================================================================

epifactors_uniprot_map = (
    epifactors_gene_annotations[
        ["gene_symbol", "UniProt_ID"]
    ]
    .loc[
        lambda df: (
            df["UniProt_ID"].notna()
            & df["UniProt_ID"].ne("#")
        )
    ]
    .drop_duplicates()
)

{
    "mapped_genes": epifactors_uniprot_map["gene_symbol"].nunique(),
    "unique_uniprot_ids": epifactors_uniprot_map["UniProt_ID"].nunique(),
    "duplicated_uniprot_ids": (
        epifactors_uniprot_map["UniProt_ID"]
        .duplicated()
        .sum()
    ),
    "examples": epifactors_uniprot_map.head(10).to_dict("records"),
}

{'mapped_genes': 796,
 'unique_uniprot_ids': 795,
 'duplicated_uniprot_ids': np.int64(3),
 'examples': [{'gene_symbol': 'A1CF', 'UniProt_ID': 'A1CF_HUMAN'},
  {'gene_symbol': 'ACINU', 'UniProt_ID': 'ACINU_HUMAN'},
  {'gene_symbol': 'ACTB', 'UniProt_ID': 'ACTB_HUMAN'},
  {'gene_symbol': 'ACTL6A', 'UniProt_ID': 'ACL6A_HUMAN'},
  {'gene_symbol': 'ACTL6B', 'UniProt_ID': 'ACL6B_HUMAN'},
  {'gene_symbol': 'ACTR3B', 'UniProt_ID': 'ARP3B_HUMAN'},
  {'gene_symbol': 'ACTR5', 'UniProt_ID': 'ARP5_HUMAN'},
  {'gene_symbol': 'ACTR6', 'UniProt_ID': 'ARP6_HUMAN'},
  {'gene_symbol': 'ACTR8', 'UniProt_ID': 'ARP8_HUMAN'},
  {'gene_symbol': 'ADNP', 'UniProt_ID': 'ADNP_HUMAN'}]}

In [16]:
# =============================================================================
# Inspect ambiguous UniProt-to-HGNC mappings
# =============================================================================

ambiguous_uniprot_ids = (
    epifactors_uniprot_map.loc[
        epifactors_uniprot_map["UniProt_ID"].duplicated(keep=False),
        "UniProt_ID",
    ]
    .unique()
)

epifactors_uniprot_map.loc[
    epifactors_uniprot_map["UniProt_ID"].isin(ambiguous_uniprot_ids)
].sort_values(
    ["UniProt_ID", "gene_symbol"]
)

,gene_symbol,UniProt_ID
114,CDY1,CDY1_HUMAN
115,CDY1B,CDY1_HUMAN
116,CDY2A,CDY2_HUMAN
117,CDY2B,CDY2_HUMAN
293,HSPA1A,HSP71_HUMAN
295,HSPA1B,HSP71_HUMAN


In [17]:
# =============================================================================
# Assess relevance of ambiguous UniProt mappings
# =============================================================================

ambiguous_mapping_relevance = []

for uniprot_id in ambiguous_uniprot_ids:
    mapped_genes = sorted(
        epifactors_uniprot_map.loc[
            epifactors_uniprot_map["UniProt_ID"].eq(uniprot_id),
            "gene_symbol",
        ]
    )

    ambiguous_mapping_relevance.append({
        "UniProt_ID": uniprot_id,
        "mapped_genes": ", ".join(mapped_genes),
        "genes_in_consensus_universe": ", ".join(
            gene for gene in mapped_genes
            if gene in consensus_gene_symbols
        ),
        "complex_rows_containing_id": (
            epifactors_complexes["UniProt_ID"]
            .fillna("")
            .str.contains(uniprot_id, regex=False)
            .sum()
        ),
    })

pd.DataFrame(ambiguous_mapping_relevance)

,UniProt_ID,mapped_genes,genes_in_consensus_universe,complex_rows_containing_id
0,CDY1_HUMAN,"CDY1, CDY1B",,0
1,CDY2_HUMAN,"CDY2A, CDY2B",,0
2,HSP71_HUMAN,"HSPA1A, HSPA1B",,3


In [18]:
# =============================================================================
# Reconstruct unambiguous EpiFactors complex membership
# =============================================================================

unambiguous_uniprot_map = epifactors_uniprot_map.loc[
    ~epifactors_uniprot_map["UniProt_ID"].isin(ambiguous_uniprot_ids)
].copy()

complex_membership = (
    epifactors_complexes[
        ["Group_name", "Complex_name", "UniProt_ID"]
    ]
    .assign(
        member_uniprot_id=lambda df: (
            df["UniProt_ID"]
            .fillna("")
            .str.replace(r"[()]", "", regex=True)
            .str.split(r"\s*,\s*|\|", regex=True)
        )
    )
    .explode("member_uniprot_id")
)

complex_membership["member_uniprot_id"] = (
    complex_membership["member_uniprot_id"].str.strip()
)

complex_membership = (
    complex_membership.loc[
        complex_membership["member_uniprot_id"].ne("")
    ]
    .merge(
        unambiguous_uniprot_map,
        left_on="member_uniprot_id",
        right_on="UniProt_ID",
        how="inner",
    )
    .drop_duplicates(
        ["Group_name", "Complex_name", "gene_symbol"]
    )
)

In [19]:
# =============================================================================
# Quantify exploratory complex coverage
# =============================================================================

complex_gene_sets = {}
complex_coverage_records = []

for (group_name, complex_name), members in complex_membership.groupby(
    ["Group_name", "Complex_name"]
):
    source_genes = set(members["gene_symbol"])
    genes_in_universe = source_genes & consensus_gene_symbols

    complex_gene_sets[(group_name, complex_name)] = genes_in_universe

    complex_coverage_records.append({
        "group_name": group_name,
        "complex_name": complex_name,
        "source_gene_count": len(source_genes),
        "genes_in_consensus_universe": len(genes_in_universe),
        "coverage_fraction": (
            len(genes_in_universe) / len(source_genes)
        ),
        "eligible_for_enrichment": (
            len(genes_in_universe) >= MIN_SET_SIZE
        ),
    })

complex_coverage = (
    pd.DataFrame(complex_coverage_records)
    .sort_values(
        ["genes_in_consensus_universe", "source_gene_count"],
        ascending=False,
    )
    .reset_index(drop=True)
)

complex_coverage.head(20)

,group_name,complex_name,source_gene_count,genes_in_consensus_universe,coverage_fraction,eligible_for_enrichment
0,SWI/SNF,SWI/SNF BRM-BRG1,20,1,0.050000,False
1,PcG and PcG-like,PRC1,18,1,0.055556,False
2,SWI/SNF,PBAF,14,1,0.071429,False
3,SWI/SNF,SWI/SNF_Brm,14,1,0.071429,False
4,SWI/SNF,nBAF,14,1,0.071429,False
5,SWI/SNF,BAF,13,1,0.076923,False
6,SWI/SNF,SWI/SNF_Brg1(I),11,1,0.090909,False
7,SWI/SNF,SWI/SNF_Brg1(II),8,1,0.125000,False
8,HAT,NuA4,17,0,0.000000,False
9,HDAC,NuRD,14,0,0.000000,False


In [20]:
# =============================================================================
# Rank consensus and source-system gene weights
# =============================================================================

ranked_consensus_gene_weights = consensus_gene_weights.copy()

ranked_consensus_gene_weights["absolute_consensus_weight"] = (
    ranked_consensus_gene_weights["consensus_weight"].abs()
)

ranked_consensus_gene_weights["oriented_cell_line_loading"] = (
    ranked_consensus_gene_weights["cell_line_loading"]
    * ranked_consensus_gene_weights["orientation_multiplier"]
)

for value_column, rank_prefix in [
    ("consensus_weight", "consensus"),
    ("tumor_loading", "tumor"),
    ("oriented_cell_line_loading", "cell_line"),
]:
    absolute_values = ranked_consensus_gene_weights[value_column].abs()

    ranked_consensus_gene_weights[
        f"{rank_prefix}_abs_rank"
    ] = absolute_values.groupby(
        ranked_consensus_gene_weights["consensus_program_id"]
    ).rank(
        method="average",
        ascending=False,
    )

    ranked_consensus_gene_weights[
        f"{rank_prefix}_abs_rank_percentile"
    ] = 1 - (
        ranked_consensus_gene_weights[f"{rank_prefix}_abs_rank"] - 1
    ) / (
        ranked_consensus_gene_weights.groupby(
            "consensus_program_id"
        )["gene_symbol"].transform("size") - 1
    )

In [21]:
# =============================================================================
# Define empirical rank-enrichment test
# =============================================================================

def evaluate_rank_enrichment(program_weights, gene_set, rng):
    """Test concentration of a gene set toward high absolute consensus weights."""
    selected = program_weights["gene_symbol"].isin(gene_set)

    observed_percentiles = program_weights.loc[
        selected,
        "consensus_abs_rank_percentile",
    ].to_numpy()

    universe_percentiles = program_weights[
        "consensus_abs_rank_percentile"
    ].to_numpy()

    set_size = observed_percentiles.size
    observed_mean = observed_percentiles.mean()

    null_means = np.fromiter(
        (
            rng.choice(
                universe_percentiles,
                size=set_size,
                replace=False,
            ).mean()
            for _ in range(N_PERMUTATIONS)
        ),
        dtype=float,
        count=N_PERMUTATIONS,
    )

    selected_weights = program_weights.loc[selected]

    return {
        "mean_abs_rank_percentile": observed_mean,
        "abs_rank_shift_from_null": observed_mean - null_means.mean(),
        "empirical_p_value": (
            np.count_nonzero(null_means >= observed_mean) + 1
        ) / (N_PERMUTATIONS + 1),
        "mean_signed_consensus_weight": (
            selected_weights["consensus_weight"].mean()
        ),
        "positive_weight_fraction": (
            selected_weights["consensus_weight"].gt(0).mean()
        ),
        "negative_weight_fraction": (
            selected_weights["consensus_weight"].lt(0).mean()
        ),
        "tumor_mean_abs_rank_percentile": (
            selected_weights["tumor_abs_rank_percentile"].mean()
        ),
        "cell_line_mean_abs_rank_percentile": (
            selected_weights["cell_line_abs_rank_percentile"].mean()
        ),
    }

In [22]:
# =============================================================================
# Run primary epigenetic-regulator enrichment
# =============================================================================

rng = np.random.default_rng(RANDOM_SEED)

eligible_primary_classes = (
    primary_regulator_class_coverage.loc[
        primary_regulator_class_coverage["eligible_for_enrichment"],
        "annotation_name",
    ]
    .sort_values()
    .tolist()
)

primary_enrichment_records = []

for program_id in sorted(
    ranked_consensus_gene_weights["consensus_program_id"].unique()
):
    program_weights = ranked_consensus_gene_weights.loc[
        ranked_consensus_gene_weights["consensus_program_id"].eq(program_id)
    ]

    for class_name in eligible_primary_classes:
        coverage = primary_regulator_class_coverage.loc[
            primary_regulator_class_coverage["annotation_name"].eq(class_name)
        ].iloc[0]

        result = evaluate_rank_enrichment(
            program_weights,
            primary_regulator_gene_sets[class_name],
            rng,
        )

        primary_enrichment_records.append({
            "consensus_program_id": program_id,
            "annotation_level": "regulator_class",
            "annotation_name": class_name,
            "analysis_tier": "primary",
            "source_gene_count": coverage["source_gene_count"],
            "genes_in_consensus_universe": coverage[
                "genes_in_consensus_universe"
            ],
            "coverage_fraction": coverage["coverage_fraction"],
            "eligible_for_enrichment": True,
            **result,
        })

primary_enrichment_results = pd.DataFrame(
    primary_enrichment_records
)

primary_enrichment_results

,consensus_program_id,annotation_level,annotation_name,analysis_tier,source_gene_count,genes_in_consensus_universe,coverage_fraction,eligible_for_enrichment,mean_abs_rank_percentile,abs_rank_shift_from_null,empirical_p_value,mean_signed_consensus_weight,positive_weight_fraction,negative_weight_fraction,tumor_mean_abs_rank_percentile,cell_line_mean_abs_rank_percentile
0,CONSENSUS_TX_01,regulator_class,EPIFACTORS_ALL,primary,796,32,0.040201,True,0.477256,-0.023398,0.677532,0.006899,0.6250,0.3750,0.479847,0.489793
1,CONSENSUS_TX_02,regulator_class,EPIFACTORS_ALL,primary,796,32,0.040201,True,0.476078,-0.023445,0.672233,0.001704,0.5625,0.4375,0.517313,0.500445
2,CONSENSUS_TX_03,regulator_class,EPIFACTORS_ALL,primary,796,32,0.040201,True,0.562801,0.062722,0.109889,-0.009365,0.2500,0.7500,0.473265,0.441962


In [23]:
# =============================================================================
# Apply multiple-testing correction to primary enrichment
# =============================================================================

primary_enrichment_results["bh_q_value"] = multipletests(
    primary_enrichment_results["empirical_p_value"],
    method="fdr_bh",
)[1]

primary_enrichment_results[
    [
        "consensus_program_id",
        "annotation_name",
        "mean_abs_rank_percentile",
        "abs_rank_shift_from_null",
        "empirical_p_value",
        "bh_q_value",
    ]
]

,consensus_program_id,annotation_name,mean_abs_rank_percentile,abs_rank_shift_from_null,empirical_p_value,bh_q_value
0,CONSENSUS_TX_01,EPIFACTORS_ALL,0.477256,-0.023398,0.677532,0.677532
1,CONSENSUS_TX_02,EPIFACTORS_ALL,0.476078,-0.023445,0.672233,0.677532
2,CONSENSUS_TX_03,EPIFACTORS_ALL,0.562801,0.062722,0.109889,0.329667


In [24]:
# =============================================================================
# Define descriptive gene-set summary
# =============================================================================

def summarize_gene_set_context(program_weights, gene_set):
    """Summarize rank and direction context without inferential testing."""
    selected_weights = program_weights.loc[
        program_weights["gene_symbol"].isin(gene_set)
    ]

    if selected_weights.empty:
        return {
            "mean_abs_rank_percentile": np.nan,
            "abs_rank_shift_from_null": np.nan,
            "mean_signed_consensus_weight": np.nan,
            "positive_weight_fraction": np.nan,
            "negative_weight_fraction": np.nan,
            "tumor_mean_abs_rank_percentile": np.nan,
            "cell_line_mean_abs_rank_percentile": np.nan,
        }

    return {
        "mean_abs_rank_percentile": (
            selected_weights["consensus_abs_rank_percentile"].mean()
        ),
        "abs_rank_shift_from_null": np.nan,
        "mean_signed_consensus_weight": (
            selected_weights["consensus_weight"].mean()
        ),
        "positive_weight_fraction": (
            selected_weights["consensus_weight"].gt(0).mean()
        ),
        "negative_weight_fraction": (
            selected_weights["consensus_weight"].lt(0).mean()
        ),
        "tumor_mean_abs_rank_percentile": (
            selected_weights["tumor_abs_rank_percentile"].mean()
        ),
        "cell_line_mean_abs_rank_percentile": (
            selected_weights["cell_line_abs_rank_percentile"].mean()
        ),
    }

In [25]:
# =============================================================================
# Summarize non-eligible primary regulator classes descriptively
# =============================================================================

descriptive_primary_records = []

noneligible_primary_classes = (
    primary_regulator_class_coverage.loc[
        ~primary_regulator_class_coverage["eligible_for_enrichment"],
        "annotation_name",
    ]
    .sort_values()
    .tolist()
)

for program_id in sorted(
    ranked_consensus_gene_weights["consensus_program_id"].unique()
):
    program_weights = ranked_consensus_gene_weights.loc[
        ranked_consensus_gene_weights["consensus_program_id"].eq(program_id)
    ]

    for class_name in noneligible_primary_classes:
        coverage = primary_regulator_class_coverage.loc[
            primary_regulator_class_coverage["annotation_name"].eq(class_name)
        ].iloc[0]

        context = summarize_gene_set_context(
            program_weights,
            primary_regulator_gene_sets[class_name],
        )

        descriptive_primary_records.append({
            "consensus_program_id": program_id,
            "annotation_level": "regulator_class",
            "annotation_name": class_name,
            "analysis_tier": "primary_descriptive",
            "source_gene_count": coverage["source_gene_count"],
            "genes_in_consensus_universe": coverage[
                "genes_in_consensus_universe"
            ],
            "coverage_fraction": coverage["coverage_fraction"],
            "eligible_for_enrichment": False,
            **context,
            "empirical_p_value": np.nan,
            "bh_q_value": np.nan,
        })

descriptive_primary_results = pd.DataFrame(
    descriptive_primary_records
)

descriptive_primary_results[
    [
        "consensus_program_id",
        "annotation_name",
        "genes_in_consensus_universe",
        "mean_abs_rank_percentile",
    ]
]

,consensus_program_id,annotation_name,genes_in_consensus_universe,mean_abs_rank_percentile
0,CONSENSUS_TX_01,CHROMATIN_REMODELING_SYSTEM,7,0.478284
1,CONSENSUS_TX_01,DNA_MODIFICATION,1,0.587521
2,CONSENSUS_TX_01,HISTONE_ACETYLATION,2,0.341290
3,CONSENSUS_TX_01,HISTONE_METHYLATION,4,0.592546
4,CONSENSUS_TX_01,POLYCOMB_GROUP,0,NaN
5,CONSENSUS_TX_02,CHROMATIN_REMODELING_SYSTEM,7,0.336624
6,CONSENSUS_TX_02,DNA_MODIFICATION,1,0.459799
7,CONSENSUS_TX_02,HISTONE_ACETYLATION,2,0.362856
8,CONSENSUS_TX_02,HISTONE_METHYLATION,4,0.506491
9,CONSENSUS_TX_02,POLYCOMB_GROUP,0,NaN


In [26]:
# =============================================================================
# Summarize EpiFactors complexes descriptively
# =============================================================================

descriptive_complex_records = []

for program_id in sorted(
    ranked_consensus_gene_weights["consensus_program_id"].unique()
):
    program_weights = ranked_consensus_gene_weights.loc[
        ranked_consensus_gene_weights["consensus_program_id"].eq(program_id)
    ]

    for _, coverage in complex_coverage.iterrows():
        group_name = coverage["group_name"]
        complex_name = coverage["complex_name"]
        gene_set = complex_gene_sets[(group_name, complex_name)]

        context = summarize_gene_set_context(
            program_weights,
            gene_set,
        )

        descriptive_complex_records.append({
            "consensus_program_id": program_id,
            "annotation_level": "protein_complex",
            "annotation_name": f"{group_name}::{complex_name}",
            "analysis_tier": "exploratory_descriptive",
            "source_gene_count": coverage["source_gene_count"],
            "genes_in_consensus_universe": coverage[
                "genes_in_consensus_universe"
            ],
            "coverage_fraction": coverage["coverage_fraction"],
            "eligible_for_enrichment": False,
            **context,
            "empirical_p_value": np.nan,
            "bh_q_value": np.nan,
        })

descriptive_complex_results = pd.DataFrame(
    descriptive_complex_records
)

descriptive_complex_results[
    [
        "consensus_program_id",
        "annotation_name",
        "genes_in_consensus_universe",
        "mean_abs_rank_percentile",
    ]
].head(20)

,consensus_program_id,annotation_name,genes_in_consensus_universe,mean_abs_rank_percentile
0,CONSENSUS_TX_01,SWI/SNF::SWI/SNF BRM-BRG1,1,0.077052
1,CONSENSUS_TX_01,PcG and PcG-like::PRC1,1,0.176717
2,CONSENSUS_TX_01,SWI/SNF::PBAF,1,0.077052
3,CONSENSUS_TX_01,SWI/SNF::SWI/SNF_Brm,1,0.077052
4,CONSENSUS_TX_01,SWI/SNF::nBAF,1,0.077052
5,CONSENSUS_TX_01,SWI/SNF::BAF,1,0.077052
6,CONSENSUS_TX_01,SWI/SNF::SWI/SNF_Brg1(I),1,0.077052
7,CONSENSUS_TX_01,SWI/SNF::SWI/SNF_Brg1(II),1,0.077052
8,CONSENSUS_TX_01,HAT::NuA4,0,NaN
9,CONSENSUS_TX_01,HDAC::NuRD,0,NaN


In [27]:
# =============================================================================
# Consolidate EpiFactors annotations by HGNC gene
# =============================================================================

def collapse_annotation_values(series):
    """Join unique non-empty EpiFactors annotations without adding mappings."""
    values = [
        str(value).strip()
        for value in series.dropna()
        if str(value).strip() not in {"", "#"}
    ]
    return " | ".join(dict.fromkeys(values))


epifactors_gene_context = (
    epifactors_gene_annotations.loc[
        epifactors_gene_annotations["in_consensus_universe"],
        [
            "gene_symbol",
            "GeneTag",
            "GeneDesc",
            "Function",
            "Modification",
            "Complex_name",
        ],
    ]
    .groupby("gene_symbol", as_index=False)
    .agg({
        "GeneTag": collapse_annotation_values,
        "GeneDesc": collapse_annotation_values,
        "Function": collapse_annotation_values,
        "Modification": collapse_annotation_values,
        "Complex_name": collapse_annotation_values,
    })
)

epifactors_gene_context.shape

(32, 6)

In [28]:
# =============================================================================
# Assign primary regulator classes to EpiFactors genes
# =============================================================================

specific_primary_classes = [
    class_name
    for class_name in PRIMARY_REGULATOR_CLASSES
    if class_name != "EPIFACTORS_ALL"
]

epifactors_gene_context["primary_regulator_classes"] = (
    epifactors_gene_context["gene_symbol"].map(
        lambda gene: " | ".join(
            class_name
            for class_name in specific_primary_classes
            if gene in primary_regulator_gene_sets[class_name]
        )
    )
)

In [29]:
# =============================================================================
# Build regulator gene-level consensus context
# =============================================================================

regulator_gene_context = (
    ranked_consensus_gene_weights.loc[
        ranked_consensus_gene_weights["gene_symbol"].isin(
            epifactors_gene_context["gene_symbol"]
        ),
        [
            "consensus_program_id",
            "gene_symbol",
            "consensus_weight",
            "absolute_consensus_weight",
            "consensus_abs_rank",
            "consensus_abs_rank_percentile",
            "tumor_loading",
            "tumor_abs_rank_percentile",
            "oriented_cell_line_loading",
            "cell_line_abs_rank_percentile",
        ],
    ]
    .merge(
        epifactors_gene_context,
        on="gene_symbol",
        how="left",
    )
    .sort_values(
        ["consensus_program_id", "consensus_abs_rank"]
    )
    .reset_index(drop=True)
)

regulator_gene_context.shape

(96, 16)

In [30]:
# =============================================================================
# Annotate consensus-weight direction
# =============================================================================

regulator_gene_context["weight_direction"] = np.select(
    [
        regulator_gene_context["consensus_weight"].gt(0),
        regulator_gene_context["consensus_weight"].lt(0),
    ],
    [
        "positive",
        "negative",
    ],
    default="zero",
)

In [31]:
# =============================================================================
# Inspect family-relevant EpiFactors gene annotations
# =============================================================================

epifactors_gene_context[
    [
        "gene_symbol",
        "GeneTag",
        "Function",
        "Modification",
    ]
].sort_values("gene_symbol")

,gene_symbol,GeneTag,Function,Modification
0,ACTL6B,,Chromatin remodeling cofactor,
1,APOBEC3B,APOBEC,"DNA modification, RNA modification","DNA demethylation, mRNA editing"
2,CBX2,,Histone modification read,
3,CELF3,RBM,RNA modification,Alternative splicing
4,CELF4,RBM,RNA modification,Alternative splicing
5,EYA1,PTPE,Histone modification erase,Histone phosphorylation
6,EYA2,PTPE,Histone modification erase,Histone phosphorylation
7,EYA4,PTPE,Histone modification erase,Histone phosphorylation
8,FOXA1,FOX,"Chromatin remodeling, TF",
9,GADD45G,,Chromatin remodeling,


In [32]:
# =============================================================================
# Annotate descriptive DNMT/TET/HDAC/KDM families
# =============================================================================

FAMILY_PATTERNS = {
    "DNMT": r"^DNMT",
    "TET": r"^TET",
    "HDAC": r"^HDAC",
    "KDM": r"^KDM",
}

regulator_gene_context["canonical_family"] = ""

for family_name, pattern in FAMILY_PATTERNS.items():
    regulator_gene_context.loc[
        regulator_gene_context["gene_symbol"].str.match(pattern),
        "canonical_family",
    ] = family_name

(
    regulator_gene_context.loc[
        regulator_gene_context["canonical_family"].ne(""),
        ["canonical_family", "gene_symbol"],
    ]
    .drop_duplicates()
    .sort_values(["canonical_family", "gene_symbol"])
)

,canonical_family,gene_symbol
11,HDAC,HDAC9
5,KDM,KDM5D


In [33]:
# =============================================================================
# Inspect descriptive HDAC/KDM family context
# =============================================================================

descriptive_family_context = (
    regulator_gene_context.loc[
        regulator_gene_context["canonical_family"].ne(""),
        [
            "consensus_program_id",
            "canonical_family",
            "gene_symbol",
            "consensus_weight",
            "consensus_abs_rank",
            "consensus_abs_rank_percentile",
            "tumor_loading",
            "tumor_abs_rank_percentile",
            "oriented_cell_line_loading",
            "cell_line_abs_rank_percentile",
            "weight_direction",
        ],
    ]
    .sort_values(
        [
            "canonical_family",
            "gene_symbol",
            "consensus_program_id",
        ]
    )
    .reset_index(drop=True)
)

descriptive_family_context

,consensus_program_id,canonical_family,gene_symbol,consensus_weight,consensus_abs_rank,consensus_abs_rank_percentile,tumor_loading,tumor_abs_rank_percentile,oriented_cell_line_loading,cell_line_abs_rank_percentile,weight_direction
0,CONSENSUS_TX_01,HDAC,HDAC9,0.011164,897.0,0.624791,0.042924,0.468593,0.058640,0.097571,positive
1,CONSENSUS_TX_02,HDAC,HDAC9,-0.001222,2248.0,0.059045,0.044631,0.446817,-0.175503,0.570352,negative
2,CONSENSUS_TX_03,HDAC,HDAC9,0.016857,837.0,0.649916,0.069141,0.512982,0.186193,0.856784,positive
3,CONSENSUS_TX_01,KDM,KDM5D,0.018647,431.0,0.819933,0.018808,0.216499,0.344904,0.742044,positive
4,CONSENSUS_TX_02,KDM,KDM5D,0.012588,1143.0,0.521776,0.113791,0.841290,-0.089602,0.299832,positive
5,CONSENSUS_TX_03,KDM,KDM5D,-0.011112,1306.0,0.453518,-0.016167,0.131072,-0.029990,0.247906,negative


In [34]:
# =============================================================================
# Integrate epigenetic-regulator enrichment summary
# =============================================================================

epigenetic_regulator_enrichment_summary = (
    pd.concat(
        [
            primary_enrichment_results,
            descriptive_primary_results,
            descriptive_complex_results,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "consensus_program_id",
            "annotation_level",
            "annotation_name",
        ]
    )
    .reset_index(drop=True)
)

{
    "shape": epigenetic_regulator_enrichment_summary.shape,
    "analysis_tiers": (
        epigenetic_regulator_enrichment_summary["analysis_tier"]
        .value_counts()
        .to_dict()
    ),
    "eligible_tests": int(
        epigenetic_regulator_enrichment_summary[
            "eligible_for_enrichment"
        ].sum()
    ),
}

{'shape': (237, 17),
 'analysis_tiers': {'exploratory_descriptive': 219,
  'primary_descriptive': 15,
  'primary': 3},
 'eligible_tests': 3}

In [35]:
# =============================================================================
# Finalize epigenetic-regulator gene context
# =============================================================================

epigenetic_regulator_gene_context = (
    regulator_gene_context[
        [
            "consensus_program_id",
            "gene_symbol",
            "consensus_weight",
            "absolute_consensus_weight",
            "consensus_abs_rank",
            "consensus_abs_rank_percentile",
            "tumor_loading",
            "tumor_abs_rank_percentile",
            "oriented_cell_line_loading",
            "cell_line_abs_rank_percentile",
            "weight_direction",
            "GeneTag",
            "GeneDesc",
            "Function",
            "Modification",
            "Complex_name",
            "primary_regulator_classes",
            "canonical_family",
        ]
    ]
    .rename(
        columns={
            "GeneTag": "epifactors_gene_tag",
            "GeneDesc": "epifactors_gene_description",
            "Function": "epifactors_function",
            "Modification": "epifactors_modification",
            "Complex_name": "epifactors_complex_name",
        }
    )
    .reset_index(drop=True)
)

epigenetic_regulator_gene_context.shape

(96, 18)

In [36]:
# =============================================================================
# Define notebook 403 output paths
# =============================================================================

ENRICHMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "403_epigenetic_regulator_enrichment_summary.csv"
)

GENE_CONTEXT_PATH = (
    OUTPUT_DIR
    / "403_epigenetic_regulator_gene_context.csv"
)

In [37]:
# =============================================================================
# Write notebook 403 analytical outputs
# =============================================================================

epigenetic_regulator_enrichment_summary.to_csv(
    ENRICHMENT_SUMMARY_PATH,
    index=False,
)

epigenetic_regulator_gene_context.to_csv(
    GENE_CONTEXT_PATH,
    index=False,
)

In [38]:
# =============================================================================
# Verify notebook 403 published outputs
# =============================================================================

published_outputs = {
    ENRICHMENT_SUMMARY_PATH.name: pd.read_csv(
        ENRICHMENT_SUMMARY_PATH
    ).shape,
    GENE_CONTEXT_PATH.name: pd.read_csv(
        GENE_CONTEXT_PATH
    ).shape,
}

{
    "published_artifact_count": len(published_outputs),
    "published_outputs": published_outputs,
}

{'published_artifact_count': 2,
 'published_outputs': {'403_epigenetic_regulator_enrichment_summary.csv': (237,
   17),
  '403_epigenetic_regulator_gene_context.csv': (96, 18)}}

## Final interpretation

Notebook 403 evaluated whether curated EpiFactors v2.1 epigenetic regulators are
preferentially concentrated among high-weight genes of the three frozen
cross-system consensus transcriptomic representations.

The analysis was restricted to the frozen 2,389-gene consensus universe. Of the
796 unique EpiFactors genes represented in the protein-level annotation table,
32 were present in this universe.

Under the prespecified minimum set size of 10 genes, only the global
`EPIFACTORS_ALL` regulator set was eligible for inferential enrichment testing.
The more specific DNA-modification, histone-acetylation, histone-methylation,
chromatin-remodeling, and Polycomb classes contained between 0 and 7 genes in
the frozen universe and were therefore retained as descriptive context only.
No EpiFactors protein complex reached the minimum eligible size; exploratory
complex annotations were likewise retained descriptively without inferential
testing.

The global EpiFactors set showed no supported enrichment in any of the three
consensus programs after empirical rank-based testing and Benjamini-Hochberg
correction:

- `CONSENSUS_TX_01`: empirical p = 0.678, BH q = 0.678
- `CONSENSUS_TX_02`: empirical p = 0.672, BH q = 0.678
- `CONSENSUS_TX_03`: empirical p = 0.110, BH q = 0.330

`CONSENSUS_TX_03` showed the largest positive shift in absolute-rank percentile,
but this pattern does not provide inferential support for enrichment.

Within the prespecified DNMT, TET, HDAC, and KDM descriptive families, only
`HDAC9` and `KDM5D` were represented in the frozen consensus universe. Their
program-specific weights and source-system ranks are therefore treated solely
as gene-level contextual information.

Overall, notebook 403 does not support broad preferential concentration of
curated epigenetic regulators within the frozen consensus representations.
More specific regulator classes and complexes cannot be evaluated
inferentially because of limited overlap with the consensus feature space.
These results therefore provide biological annotation and candidate
associated-regulator context only; they do not establish regulator activity,
causal control, mechanistic validation, or therapeutic relevance.

## Published outputs

Notebook 403 publishes exactly two downstream-consumable artifacts under
`data/processed/consensus_programs/`:

- `403_epigenetic_regulator_enrichment_summary.csv`
- `403_epigenetic_regulator_gene_context.csv`

The published artifacts were successfully reloaded and verified with final
shapes of 237 × 17 and 96 × 18, respectively.